In [ ]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.impute import KNNImputer

train = pd.read_csv('/kaggle/input/titanic/train.csv')
test = pd.read_csv('/kaggle/input/titanic/test.csv')
submissions = pd.read_csv('/kaggle/input/titanic/gender_submission.csv')

imputer = KNNImputer(n_neighbors=33)

features = ['Age', 'Fare']

train[features] = imputer.fit_transform(train[features])
test[features] = imputer.transform(test[features])

train.drop('Cabin', axis=1, inplace=True)
test.drop('Cabin', axis=1, inplace=True)

train.dropna(subset=['Embarked'], inplace=True)
test['Fare'].fillna(test['Fare'].median(), inplace=True)  

train = pd.concat([train, pd.get_dummies(train['Sex'], drop_first=True)], axis=1)
train = pd.concat([train, pd.get_dummies(train['Embarked'], drop_first=True)], axis=1)

test = pd.concat([test, pd.get_dummies(test['Sex'], drop_first=True)], axis=1)
test = pd.concat([test, pd.get_dummies(test['Embarked'], drop_first=True)], axis=1)

train.drop(['Sex', 'Embarked', 'Name', 'Ticket'], axis=1, inplace=True)
test.drop(['Sex', 'Embarked', 'Name', 'Ticket'], axis=1, inplace=True)

x = train.drop(['Survived'], axis=1)
y = train['Survived']

x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.3, random_state=42)

logmodel = LogisticRegression(max_iter=200)
logmodel.fit(x_train, y_train)

predictions = logmodel.predict(x_val)
test_predictions = logmodel.predict(test)

output = pd.DataFrame({'PassengerId': test.PassengerId,
                       'Survived': test_predictions})
output.to_csv('submission1.csv', index=False)